In [1]:
import os, json, math, textwrap, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

root=Path('/workspace')
files={p.name:p for p in root.iterdir() if p.is_file()}
for name in sorted(files):
    print(name, files[name].stat().st_size)

pred=pd.read_csv(root/'round2-frozen-predictions.csv')
key=pd.read_csv(root/'round2-print-key.csv')
r2=pd.read_csv(root/'round2-measured-drop-results.csv')
r2drop=pd.read_csv(root/'round2-per-drop-metrics.csv')
r1=pd.read_csv(root/'sobol-round1-drop-results.csv')
r1drop=pd.read_csv(root/'sobol-round1-per-drop-metrics.csv')
obj=pd.read_csv(root/'objectives-mass-normalized.csv')
sens=pd.read_csv(root/'drop-count-sensitivity.csv')
r3=pd.read_csv(root/'round3-suggested-batch.csv')
r2geom=pd.read_csv(root/'round2-as-printed-designs.csv')
r1geom=pd.read_csv(root/'sobol-round1-design-table.csv')
print('\nShapes:', {k:v.shape for k,v in [('pred',pred),('key',key),('r2',r2),('r2drop',r2drop),('r1',r1),('r1drop',r1drop),('obj',obj),('sens',sens),('r3',r3)]})
print('\nPrediction columns:', pred.columns.tolist())
print('Key columns:', key.columns.tolist())
print('R2 columns:', r2.columns.tolist())
print('\nKey mapping:')
print(key[['print_id','source_trial','mass_g_with_label','pred_printed_mass_g','defects']].to_string(index=False))

.kernel_llm_logs_1.txt 0
bo-README.md 31505
drop-count-sensitivity.csv 6263
loocv-round1-fit-diagnostics.json 854
loocv-round1-fit.csv 776
objectives-mass-normalized.csv 1442
prior-edison-adversarial-objective-review.md 32742
round2-as-printed-designs.csv 2064
round2-frozen-predictions.csv 1027
round2-measured-drop-results.csv 3991
round2-per-drop-metrics.csv 11887
round2-print-key.csv 1632
round3-suggested-batch.csv 1802
sobol-round1-design-table.csv 2035
sobol-round1-drop-results.csv 3721
sobol-round1-per-drop-metrics.csv 62670
sobol-round1-print-key.csv 1710
t3_prism_bo_campaign.py 77993
t3_prism_mass_model.py 16219

Shapes: {'pred': (9, 14), 'key': (9, 11), 'r2': (9, 31), 'r2drop': (172, 5), 'r1': (9, 28), 'r1drop': (893, 5), 'obj': (17, 10), 'sens': (45, 9), 'r3': (9, 24)}

Prediction columns: ['round', 'trial_index', 'R_mm', 'H_mm', 'twist_deg', 'strut_d_mm', 'cable_d_mm', 'pred_t180_mean', 'pred_t180_sd', 'pred_e_reb_mJ_mean', 'pred_e_reb_mJ_sd', 'pred_mass_g_mean', 'pred_mass_g

In [2]:
# Reconstruct measured energy independently from mass, e_rebound, and documented drop height.
# Infer/verify the height from the provided mass-normalized objective file rather than trusting labels.
print(obj.to_string(index=False))
merged=(key.merge(pred, left_on='source_trial', right_on='trial_index', validate='one_to_one')
          .merge(r2, left_on='print_id', right_on='specimen', validate='one_to_one', suffixes=('','_meas')))
# Join published objective values only to verify energy conversion
merged=merged.merge(obj[['specimen','e_reb_mJ','e_reb_mJ_per_g']], left_on='print_id', right_on='specimen', validate='one_to_one', suffixes=('','_obj'))
merged['inferred_h_m']=merged.e_reb_mJ/1000/(merged.e_rebound_mean*merged.mass_g*1e-3*9.80665)
print('\nInferred drop heights (m):', merged.inferred_h_m.round(6).tolist())
# independently use h=0.30 m
merged['meas_e_mJ_calc']=merged.e_rebound_mean*(merged.mass_g*1e-3)*9.80665*0.30*1000
print('Max abs energy reproduction diff (mJ):', (merged.meas_e_mJ_calc-merged.e_reb_mJ).abs().max())
for metric, obs, pm, ps in [('t180','t180_mean','pred_t180_mean','pred_t180_sd'),('e_reb_mJ','meas_e_mJ_calc','pred_e_reb_mJ_mean','pred_e_reb_mJ_sd')]:
    merged[f'{metric}_resid']=merged[obs]-merged[pm]
    merged[f'{metric}_z']=merged[f'{metric}_resid']/merged[ps]

cols=['print_id','source_trial','t180_mean','pred_t180_mean','pred_t180_sd','t180_resid','t180_z','meas_e_mJ_calc','pred_e_reb_mJ_mean','pred_e_reb_mJ_sd','e_reb_mJ_resid','e_reb_mJ_z']
print('\nCalibration table:')
print(merged[cols].sort_values('print_id').to_string(index=False, float_format=lambda x:f'{x:.4f}'))

# Standard diagnostics using posterior sd of latent mean, as requested.
def diag(z, resid):
    n=len(z)
    return dict(n=n, mean_resid=np.mean(resid), median_resid=np.median(resid), rmse=np.sqrt(np.mean(resid**2)),
                mean_z=np.mean(z), sd_z=np.std(z,ddof=1), cov68=np.mean(np.abs(z)<=1), cov95=np.mean(np.abs(z)<=1.96),
                cov997=np.mean(np.abs(z)<=3), pos=np.sum(resid>0), neg=np.sum(resid<0),
                sign_p=stats.binomtest(np.sum(resid>0),n,0.5).pvalue,
                zmean_t=np.mean(z)/(np.std(z,ddof=1)/np.sqrt(n)),
                zmean_p=stats.ttest_1samp(z,0).pvalue)
for m in ['t180','e_reb_mJ']:
    d=diag(merged[m+'_z'].to_numpy(), merged[m+'_resid'].to_numpy())
    print('\n',m,d)
# exact binomial interval for empirical coverage (small n)
for m in ['t180','e_reb_mJ']:
    for q,cut in [('68',1),('95',1.96)]:
        k=int((merged[m+'_z'].abs()<=cut).sum()); ci=stats.binomtest(k,9).proportion_ci(0.95)
        print(m,q,k,'/9 coverage CI',ci.low,ci.high)


specimen spec  round  mass_g  t180_mean  e_rebound_mean  e_reb_mJ  e_reb_mJ_per_g  pareto_abs  pareto_per_g
  6lhxfy   01      1   18.50   0.893078        0.050376     13.93          0.7529        True          True
  6nheas   05      1   21.73   0.997008        0.040246     13.07          0.6015       False         False
  9hhbkp   00      1   21.62   1.018336        0.021501      6.95          0.3213       False         False
  ajhby6   07      1   20.53   1.012658        0.019880      6.10          0.2971       False          True
  autv5r   02      1   22.04   1.040430        0.026810      8.83          0.4007       False         False
  bag26v   08      1   21.42   1.061620        0.024098      7.71          0.3602       False         False
  bpx68c   S0      1   20.23   1.011072        0.020442      6.18          0.3055       False          True
  nvxsrv   04      1   20.66   1.027549        0.026564      8.20          0.3970       False         False
  r2d2c1   03      2   19.38

In [3]:
# Correct energy reconstruction: source objective uses 60-inch = 1.524 m drop height.
h=1.524
merged['meas_e_mJ_calc']=merged.e_rebound_mean*(merged.mass_g*1e-3)*9.80665*h*1000
print('Max abs energy reproduction diff (mJ):', (merged.meas_e_mJ_calc-merged.e_reb_mJ).abs().max())
for metric, obs, pm, ps in [('t180','t180_mean','pred_t180_mean','pred_t180_sd'),('e_reb_mJ','meas_e_mJ_calc','pred_e_reb_mJ_mean','pred_e_reb_mJ_sd')]:
    merged[f'{metric}_resid']=merged[obs]-merged[pm]
    merged[f'{metric}_z']=merged[f'{metric}_resid']/merged[ps]
print(merged[cols].sort_values('print_id').to_string(index=False, float_format=lambda x:f'{x:.4f}'))
for m in ['t180','e_reb_mJ']:
    d=diag(merged[m+'_z'].to_numpy(), merged[m+'_resid'].to_numpy())
    print('\n',m,d)
    for q,cut in [('68',1),('95',1.96),('99.7',3)]:
        k=int((merged[m+'_z'].abs()<=cut).sum()); ci=stats.binomtest(k,9).proportion_ci(0.95)
        print(q,k,'/9; exact 95% CI', (ci.low,ci.high))
# Compare signs and requested ranges
print('\nresidual ranges t180',merged.t180_resid.min(),merged.t180_resid.max())
print('residual ranges energy',merged.e_reb_mJ_resid.min(),merged.e_reb_mJ_resid.max())
print('energy <= predicted', (merged.e_reb_mJ_resid<=0).sum())

Max abs energy reproduction diff (mJ): 0.004833413532678854
print_id  source_trial  t180_mean  pred_t180_mean  pred_t180_sd  t180_resid  t180_z  meas_e_mJ_calc  pred_e_reb_mJ_mean  pred_e_reb_mJ_sd  e_reb_mJ_resid  e_reb_mJ_z
  r2d2c1            13     0.9942          0.9477        0.0464      0.0465  1.0011          5.9652              8.3068            3.1761         -2.3416     -0.7373
  r2d2c2            12     1.0411          0.9256        0.0945      0.1155  1.2219          5.6245              9.8960            5.0348         -4.2715     -0.8484
  r2d2c3            16     1.3337          0.9102        0.0728      0.4235  5.8167          5.5915             10.0896            5.1953         -4.4981     -0.8658
  r2d2c4            11     1.1847          0.8849        0.0749      0.2998  4.0024          6.4430             12.2474            5.7281         -5.8044     -1.0133
  r2d2c5            18     1.1516          0.9868        0.0663      0.1648  2.4849          6.7061           

In [4]:
print('R1 result specimens:', r1[['specimen','spec','n_valid','t180_mean','e_rebound_mean','mass_g']].to_string(index=False))
print('\nR1 per-drop specimens/counts:')
print(r1drop.groupby('specimen').size())
print('\nObjective counts by round:',obj.groupby('round').size().to_dict(), 'total',len(obj))
print('\nMissing from objective table:', sorted(set(r1.specimen)-set(obj.specimen)))
print('\nR2 t1000/t180 and stability metrics:')
r2x=r2.copy(); r2x['ratio']=r2x.t1000_mean/r2x.t180_mean
print(r2x[['specimen','n_valid','t180_mean','t180_sd','t1000_mean','t1000_sd','ratio','in_dv_ms_mean','in_dv_ms_sd','out_180_g_mean','out_180_g_sd','t_second_ms_mean','t_second_ms_sd','t_drift_flag','t180_slope_pct_per_drop','t180_e2e_pct']].to_string(index=False,float_format=lambda x:f'{x:.5f}'))
# all article ratios
r1x=r1.copy(); r1x['ratio']=r1x.t1000_mean/r1x.t180_mean
print('\nR1 ratios:')
print(r1x[['specimen','t180_mean','t1000_mean','ratio']].to_string(index=False,float_format=lambda x:f'{x:.5f}'))

R1 result specimens: specimen spec  n_valid  t180_mean  e_rebound_mean  mass_g
  6lhxfy   01      101   0.893078        0.050376   18.50
  6nheas   05      101   0.997008        0.040246   21.73
  9hhbkp   00      101   1.018336        0.021501   21.62
  amdjwm  NaN      101   0.980495        0.029618     NaN
  autv5r   02      103   1.040430        0.026810   22.04
  bag26v   08      101   1.061620        0.024098   21.42
  bpx68c   S0      101   1.011072        0.020442   20.23
  nvxsrv   04      101   1.027549        0.026564   20.66
  ajhby6   07      101   1.012658        0.019880   20.53

R1 per-drop specimens/counts:
specimen
6lhxfy     99
6nheas     99
9hhbkp     99
ajhby6     99
amdjwm     99
autv5r    101
bag26v     99
bpx68c     99
nvxsrv     99
dtype: int64

Objective counts by round: {1: 8, 2: 9} total 17

Missing from objective table: ['amdjwm']

R2 t1000/t180 and stability metrics:
specimen  n_valid  t180_mean  t180_sd  t1000_mean  t1000_sd   ratio  in_dv_ms_mean  in_dv_

In [5]:
# Verify article summaries against committed stabilized per-drop rows; identify denominator mismatch.
rawsum=(r2drop.groupby('specimen').agg(n_scored=('t180','size'),t180_raw_mean=('t180','mean'),t180_raw_sd=('t180','std'),e_raw_mean=('e_rebound','mean'),e_raw_sd=('e_rebound','std')).reset_index())
chk=r2.merge(rawsum,left_on='specimen',right_on='specimen')
print(chk[['specimen','n_valid','n_scored','t180_mean','t180_raw_mean','t180_sd','t180_raw_sd','e_rebound_mean','e_raw_mean','e_rebound_sd','e_raw_sd']].to_string(index=False,float_format=lambda x:f'{x:.8f}'))
print('\nMax mean diffs:',(chk.t180_mean-chk.t180_raw_mean).abs().max(),(chk.e_rebound_mean-chk.e_raw_mean).abs().max())
# Correct SEM values over the actual stabilized scored drops and central article-noise floor.
chk['t_sem_scored']=chk.t180_raw_sd/np.sqrt(chk.n_scored)
chk['t_sigma_072']=np.sqrt((.0072*chk.t180_raw_mean)**2+chk.t_sem_scored**2)
chk['e_sem_scored']=chk.e_raw_sd/np.sqrt(chk.n_scored)
chk['e_mJ']=chk.e_raw_mean*chk.mass_g*9.80665*1.524
chk['e_drop_sem_mJ']=chk.e_sem_scored*chk.mass_g*9.80665*1.524
print('\nCorrect observation uncertainty components:')
print(chk[['specimen','n_scored','t_sem_scored','t_sigma_072','e_drop_sem_mJ']].to_string(index=False,float_format=lambda x:f'{x:.6f}'))
print('t sigma 0.72% range:',chk.t_sigma_072.min(),chk.t_sigma_072.max())

specimen  n_valid  n_scored  t180_mean  t180_raw_mean    t180_sd  t180_raw_sd  e_rebound_mean  e_raw_mean  e_rebound_sd   e_raw_sd
  r2d2c1       21        19 0.99415021     0.99415021 0.00205473   0.00205473      0.02059506  0.02059506    0.00061309 0.00061309
  r2d2c2       21        19 1.04107208     1.04107208 0.01541427   0.01541427      0.02095441  0.02095441    0.00080039 0.00080039
  r2d2c3       21        19 1.33365231     1.33365231 0.01165878   0.01165878      0.02008223  0.02008223    0.00027342 0.00027342
  r2d2c4       21        19 1.18468290     1.18468290 0.00758579   0.00758579      0.02230235  0.02230235    0.00019745 0.00019745
  r2d2c5       21        19 1.15155125     1.15155125 0.00397865   0.00397865      0.01911848  0.01911848    0.00023390 0.00023390
  r2d2c6       21        19 1.05484871     1.05484871 0.00423220   0.00423220      0.01423794  0.01423794    0.00023088 0.00023088
  r2d2c7       21        19 0.94837891     0.94837891 0.00083278   0.00083278      

In [6]:
# Recompute dependence structure over all 17 mapped articles, both rebound fraction and energies.
df=obj.copy()
metrics=['e_rebound_mean','e_reb_mJ','e_reb_mJ_per_g']
def corrs(d,x='t180_mean',ys=metrics):
    rows=[]
    for y in ys:
        for typ,fun in [('Pearson',stats.pearsonr),('Spearman',stats.spearmanr),('Kendall',stats.kendalltau)]:
            z=fun(d[x],d[y]); rows.append([len(d),y,typ,z.statistic,z.pvalue])
    return pd.DataFrame(rows,columns=['n','y','type','coef','p'])
print(corrs(df).to_string(index=False,float_format=lambda x:f'{x:.4f}'))
print('\nExcluding 6lhxfy:')
print(corrs(df[df.specimen!='6lhxfy']).to_string(index=False,float_format=lambda x:f'{x:.4f}'))
print('\nRound 2 only:')
print(corrs(df[df['round']==2]).to_string(index=False,float_format=lambda x:f'{x:.4f}'))
# Leave-one-out correlation ranges
loo=[]
for s in df.specimen:
    d=df[df.specimen!=s]
    loo.append((s,stats.pearsonr(d.t180_mean,d.e_reb_mJ).statistic,stats.spearmanr(d.t180_mean,d.e_reb_mJ).statistic))
print('\nLOO energy correlations:')
print(pd.DataFrame(loo,columns=['excluded','pearson','spearman']).sort_values('pearson').to_string(index=False,float_format=lambda x:f'{x:.3f}'))
# Bootstrap confidence interval article resampling; deterministic seed; acknowledge small-n exploratory
rng=np.random.default_rng(20260825); vals=[]
a=df[['t180_mean','e_reb_mJ']].to_numpy(); n=len(a)
for _ in range(200000):
    b=a[rng.integers(0,n,n)]
    if np.std(b[:,0])>0 and np.std(b[:,1])>0: vals.append(np.corrcoef(b.T)[0,1])
print('bootstrap Pearson percentile 95%',np.quantile(vals,[.025,.975]), 'B',len(vals))

 n              y     type    coef      p
17 e_rebound_mean  Pearson -0.5505 0.0220
17 e_rebound_mean Spearman -0.4191 0.0940
17 e_rebound_mean  Kendall -0.2941 0.1089
17       e_reb_mJ  Pearson -0.5373 0.0261
17       e_reb_mJ Spearman -0.4559 0.0659
17       e_reb_mJ  Kendall -0.3382 0.0630
17 e_reb_mJ_per_g  Pearson -0.5505 0.0220
17 e_reb_mJ_per_g Spearman -0.4191 0.0940
17 e_reb_mJ_per_g  Kendall -0.2941 0.1089

Excluding 6lhxfy:
 n              y     type    coef      p
16 e_rebound_mean  Pearson -0.4042 0.1205
16 e_rebound_mean Spearman -0.3029 0.2541
16 e_rebound_mean  Kendall -0.2000 0.3057
16       e_reb_mJ  Pearson -0.4005 0.1242
16       e_reb_mJ Spearman -0.3471 0.1878
16       e_reb_mJ  Kendall -0.2500 0.1949
16 e_reb_mJ_per_g  Pearson -0.4043 0.1204
16 e_reb_mJ_per_g Spearman -0.3029 0.2541
16 e_reb_mJ_per_g  Kendall -0.2000 0.3057

Round 2 only:
 n              y     type    coef      p
 9 e_rebound_mean  Pearson -0.4357 0.2411
 9 e_rebound_mean Spearman -0.3167 0.4064


bootstrap Pearson percentile 95% [-0.78507861 -0.16645939] B 200000


In [7]:
# Geometry-shift dose-response check: does more shrink predict larger calibration error?
g=r2geom.copy()
print(g[['specimen','source_trial','scale','R_mm','R_print_mm','H_mm','H_print_mm','strut_d_mm','strut_d_print_mm','cable_d_mm','cable_d_print_mm','mass_g']].to_string(index=False,float_format=lambda x:f'{x:.4f}'))
# specimen is numeric design index, map through source trial/key
mg=merged.merge(g[['source_trial','scale','R_print_mm','H_print_mm','strut_d_print_mm','cable_d_print_mm','mass_g']],on='source_trial',suffixes=('','_solid'))
mg['shrink_pct']=(1-mg.scale)*100
mg['mass_pred_error']=mg.mass_g-mg.pred_mass_g_mean
vars=['scale','shrink_pct','R_print_mm','H_print_mm','strut_d_print_mm','cable_d_print_mm','mass_g','mass_pred_error']
rows=[]
for x in vars:
 for y in ['t180_resid','e_reb_mJ_resid']:
  for typ,fun in [('Pearson',stats.pearsonr),('Spearman',stats.spearmanr)]:
   z=fun(mg[x],mg[y]); rows.append([x,y,typ,z.statistic,z.pvalue])
print('\nResidual-vs-shift/geometry associations:')
print(pd.DataFrame(rows,columns=['x','residual','type','coef','p']).to_string(index=False,float_format=lambda x:f'{x:.3f}'))
print('\nShrink range:',mg.shrink_pct.min(),mg.shrink_pct.max())
# Compare residual correlation with measured objective relationship
print('Residual correlation t vs energy:',stats.pearsonr(mg.t180_resid,mg.e_reb_mJ_resid),stats.spearmanr(mg.t180_resid,mg.e_reb_mJ_resid))

 specimen  source_trial  scale    R_mm  R_print_mm     H_mm  H_print_mm  strut_d_mm  strut_d_print_mm  cable_d_mm  cable_d_print_mm  mass_g
        0            10 0.8779 25.0000     21.9460  60.0000     52.6710     12.0000           10.5340      3.0000            2.6340 30.9900
        1            11 0.7531 40.0000     30.1240  60.0000     45.1860     12.0000            9.0370      5.5000            4.1420 30.9100
        2            12 0.8985 25.0000     22.4630  60.0000     53.9120     12.0000           10.7820      3.0000            2.6960 31.0100
        3            13 1.0920 25.0000     27.3010  60.0000     65.5220      7.4015            8.0830      3.0000            3.2760 30.9500
        4            14 0.8159 25.0000     20.3980  60.0000     48.9550     12.0000            9.7910      5.5000            4.4880 30.8800
        5            15 0.8722 40.0000     34.8860  60.0000     52.3300     12.0000           10.4660      3.0000            2.6160 31.0000
        6           

In [8]:
# Exact empirical Pareto sets, margins, and uncertainty using corrected article-level noise.
def pareto_min(d,x,y):
    a=d[[x,y]].to_numpy(); n=len(a); keep=[]
    for i in range(n):
        dom=np.any(np.all(a<=a[i],axis=1)&np.any(a<a[i],axis=1))
        keep.append(not dom)
    return np.array(keep)
for y in ['e_reb_mJ','e_reb_mJ_per_g']:
    calc=pareto_min(df,'t180_mean',y)
    print(y,'calculated:',df.loc[calc,'specimen'].tolist(),'file:',df.loc[df['pareto_abs' if y=='e_reb_mJ' else 'pareto_per_g'],'specimen'].tolist())
# Dominance values and prior low floor.
for s in ['bpx68c','ajhby6','r2d2c1','r2d2c2','r2d2c6']:
 print(df.loc[df.specimen==s,['specimen','mass_g','t180_mean','e_reb_mJ','e_reb_mJ_per_g']].to_string(index=False))
# Parametric Monte Carlo: independent normal observation distributions. t180 sigma central floor; rebound only within-article SEM (optimistic)
# Also use 5% relative floor rebound sensitivity, reflecting prior report's provisional conservative floor.
allx=df.copy()
# derive t sem and rebound sem for each article from raw files
raw=pd.concat([r1drop,r2drop],ignore_index=True)
rs=raw.groupby('specimen').agg(n=('t180','size'),t_sd=('t180','std'),e_sd=('e_rebound','std')).reset_index()
allx=allx.merge(rs,on='specimen')
allx['t_sig']=np.sqrt((.0072*allx.t180_mean)**2+(allx.t_sd/np.sqrt(allx.n))**2)
allx['e_abs_sem']=allx.e_sd/np.sqrt(allx.n)*allx.mass_g*9.80665*1.524
allx['e_abs_sig5']=np.sqrt(allx.e_abs_sem**2+(.05*allx.e_reb_mJ)**2)
allx['e_pg_sem']=allx.e_sd/np.sqrt(allx.n)*9.80665*1.524
allx['e_pg_sig5']=np.sqrt(allx.e_pg_sem**2+(.05*allx.e_reb_mJ_per_g)**2)

def front_mc(y,ysig,B=200000,seed=1):
 rng=np.random.default_rng(seed); n=len(allx); counts=np.zeros(n)
 # chunks
 for st in range(0,B,10000):
  b=min(10000,B-st)
  X=rng.normal(allx.t180_mean.to_numpy(),allx.t_sig.to_numpy(),(b,n))
  Y=rng.normal(allx[y].to_numpy(),allx[ysig].to_numpy(),(b,n))
  # nondominated in 2D brute-force vector b,n,n
  dom=((X[:,:,None]>=X[:,None,:])&(Y[:,:,None]>=Y[:,None,:])&((X[:,:,None]>X[:,None,:])|(Y[:,:,None]>Y[:,None,:]))).any(axis=2)
  counts+=(~dom).sum(axis=0)
 return counts/B
for y,ysig in [('e_reb_mJ','e_abs_sem'),('e_reb_mJ','e_abs_sig5'),('e_reb_mJ_per_g','e_pg_sem'),('e_reb_mJ_per_g','e_pg_sig5')]:
 allx[f'pfront_{y}_{ysig}']=front_mc(y,ysig)
 print('\n',y,ysig)
 print(allx[['specimen',f'pfront_{y}_{ysig}']].sort_values(f'pfront_{y}_{ysig}',ascending=False).to_string(index=False,float_format=lambda z:f'{z:.3f}'))


e_reb_mJ calculated: ['6lhxfy', 'r2d2c1', 'r2d2c2', 'r2d2c6', 'r2d2c7'] file: ['6lhxfy', 'r2d2c1', 'r2d2c2', 'r2d2c6', 'r2d2c7']
e_reb_mJ_per_g calculated: ['6lhxfy', 'ajhby6', 'bpx68c', 'r2d2c1', 'r2d2c6', 'r2d2c7'] file: ['6lhxfy', 'ajhby6', 'bpx68c', 'r2d2c1', 'r2d2c6', 'r2d2c7']
specimen  mass_g  t180_mean  e_reb_mJ  e_reb_mJ_per_g
  bpx68c   20.23   1.011072      6.18          0.3055
specimen  mass_g  t180_mean  e_reb_mJ  e_reb_mJ_per_g
  ajhby6   20.53   1.012658       6.1          0.2971
specimen  mass_g  t180_mean  e_reb_mJ  e_reb_mJ_per_g
  r2d2c1   19.38    0.99415      5.97          0.3078
specimen  mass_g  t180_mean  e_reb_mJ  e_reb_mJ_per_g
  r2d2c2   17.96   1.041072      5.62          0.3132
specimen  mass_g  t180_mean  e_reb_mJ  e_reb_mJ_per_g
  r2d2c6   19.11   1.054849      4.07          0.2128



 e_reb_mJ e_abs_sem
specimen  pfront_e_reb_mJ_e_abs_sem
  6lhxfy                      1.000
  r2d2c6                      1.000
  r2d2c7                      1.000
  r2d2c1                      1.000
  r2d2c2                      0.890
  r2d2c9                      0.123
  bpx68c                      0.045
  ajhby6                      0.037
  9hhbkp                      0.006
  nvxsrv                      0.000
  6nheas                      0.000
  bag26v                      0.000
  autv5r                      0.000
  r2d2c5                      0.000
  r2d2c4                      0.000
  r2d2c3                      0.000
  r2d2c8                      0.000



 e_reb_mJ e_abs_sig5
specimen  pfront_e_reb_mJ_e_abs_sig5
  6lhxfy                       1.000
  r2d2c6                       1.000
  r2d2c1                       0.973
  r2d2c7                       0.972
  r2d2c2                       0.610
  ajhby6                       0.350
  bpx68c                       0.299
  r2d2c9                       0.157
  6nheas                       0.058
  9hhbkp                       0.012
  nvxsrv                       0.000
  bag26v                       0.000
  autv5r                       0.000
  r2d2c5                       0.000
  r2d2c4                       0.000
  r2d2c3                       0.000
  r2d2c8                       0.000



 e_reb_mJ_per_g e_pg_sem
specimen  pfront_e_reb_mJ_per_g_e_pg_sem
  6lhxfy                           1.000
  r2d2c6                           1.000
  r2d2c7                           1.000
  ajhby6                           1.000
  r2d2c1                           0.931
  bpx68c                           0.487
  6nheas                           0.377
  9hhbkp                           0.006
  nvxsrv                           0.000
  r2d2c2                           0.000
  autv5r                           0.000
  bag26v                           0.000
  r2d2c3                           0.000
  r2d2c5                           0.000
  r2d2c4                           0.000
  r2d2c8                           0.000
  r2d2c9                           0.000



 e_reb_mJ_per_g e_pg_sig5
specimen  pfront_e_reb_mJ_per_g_e_pg_sig5
  6lhxfy                            1.000
  r2d2c6                            1.000
  r2d2c7                            0.972
  r2d2c1                            0.952
  ajhby6                            0.595
  bpx68c                            0.434
  6nheas                            0.338
  9hhbkp                            0.102
  r2d2c2                            0.097
  r2d2c9                            0.015
  nvxsrv                            0.000
  bag26v                            0.000
  r2d2c5                            0.000
  autv5r                            0.000
  r2d2c3                            0.000
  r2d2c4                            0.000
  r2d2c8                            0.000


In [9]:
# Quantify standardized-window effects and drift handling.
print(sens.to_string(index=False,float_format=lambda x:f'{x:.6f}'))
# Directly recompute first 19 stabilized drops (same as most round 2) vs full round 1.
r1w=[]
for s,d in r1drop.groupby('specimen'):
 d=d.sort_values('drop_index')
 for col in ['t180','e_rebound']:
  full=d[col].mean(); early=d[col].iloc[:19].mean()
  r1w.append([s,col,full,early,early-full,(early/full-1)*100])
r1w=pd.DataFrame(r1w,columns=['specimen','metric','full','first19','diff','pct'])
print('\nFirst-19 vs full:')
print(r1w.to_string(index=False,float_format=lambda x:f'{x:.6f}'))
for m in ['t180','e_rebound']:
 z=r1w[r1w.metric==m]
 print(m,'max abs diff',z['diff'].abs().max(),'max abs %',z.pct.abs().max(),'mean signed %',z.pct.mean())
# drift uncertainty for c2
c2=chk[chk.specimen=='r2d2c2'].iloc[0]
drift_half=.5*.035*c2.t180_mean
sig=np.sqrt((.0072*c2.t180_mean)**2+c2.t_sem_scored**2+drift_half**2)
print('\nr2d2c2 drift half excursion',drift_half,'augmented sigma',sig)
# Linear regressions per R2 drop using committed rows, report slope and tests; especially anomaly vs drift
for s,d in r2drop.groupby('specimen'):
 z=stats.linregress(d.drop_index,d.t180)
 if s in ['r2d2c2','r2d2c3']:
  print(s,'t180 slope',z.slope,'per drop percent',100*z.slope/d.t180.mean(),'p',z.pvalue,'r',z.rvalue,
       'first5',d.sort_values('drop_index').t180.iloc[:5].mean(),'last5',d.sort_values('drop_index').t180.iloc[-5:].mean())


specimen  n_drops   window  t180_mean  t180_sem  t180_dev_from_full  e_rebound_mean  e_rebound_sem  e_rebound_dev_pct
  6lhxfy       10 first-10   0.885898  0.001000           -0.007180        0.049130       0.000139          -2.474172
  6lhxfy       20 first-20   0.889685  0.001026           -0.003393        0.049516       0.000119          -1.707258
  6lhxfy       30 first-30   0.892165  0.000953           -0.000913        0.049844       0.000118          -1.056514
  6lhxfy       50 first-50   0.894342  0.000709            0.001264        0.050269       0.000130          -0.213783
  6lhxfy       99     full   0.893078  0.000418            0.000000        0.050376       0.000106           0.000000
  6nheas       10 first-10   0.994541  0.001279           -0.002467        0.039816       0.000086          -1.068439
  6nheas       20 first-20   0.994886  0.000774           -0.002123        0.039941       0.000063          -0.757402
  6nheas       30 first-30   0.994651  0.000651         

In [10]:
# Round-3 candidate audit: geometry, boundary frequencies, novelty/distances to R2 low-rebound anchor.
print(r3.to_string(index=False,float_format=lambda x:f'{x:.4f}'))
for c in ['R_mm','H_mm','twist_deg','strut_d_mm','cable_d_mm']:
 print(c, r3[c].value_counts().sort_index().to_dict())
print('\nPrediction summaries:')
print(r3[['pred_t180_mean','pred_t180_sd','pred_e_reb_mJ_mean','pred_e_reb_mJ_sd','scale','solid_mass_g']].describe().to_string())
# normalized Euclidean distances to all training base shapes reconstructed from obj via design tables/key.
# Map r1 geometry from design table and records; inspect schema/mappings.
print('\nR1 geom cols',r1geom.columns.tolist())
print(r1geom[['specimen','R_mm','H_mm','twist_deg','strut_d_mm','cable_d_mm','scale','R_print_mm','H_print_mm','strut_d_print_mm','cable_d_print_mm']].to_string(index=False))
# Trial candidates that violate constraints
print('\nConstraint failures:',r3.loc[(~r3.envelope_ok)|(~r3.cable_bridge_ok),['trial_index','envelope_ok','cable_bridge_ok']].to_dict('records'))
# How much round3 projection scales and whether constant-print mass is represented by base shape + mass only
print('Round3 scale range',r3.scale.min(),r3.scale.max())

 round  trial_index    R_mm     H_mm  twist_deg  strut_d_mm  cable_d_mm  mass_printed_g  pred_t180_mean  pred_t180_sd  pred_e_reb_mJ_mean  pred_e_reb_mJ_sd  pred_e_rebound_approx  target_mass_g  scale  R_print_mm  H_print_mm  strut_d_print_mm  cable_d_print_mm  joint_d_print_mm  solid_mass_g  envelope_cm3  envelope_ok  cable_bridge_ok
     2           19 30.5204 110.0000    40.0000     12.0000      5.5000         20.2400          0.9978        0.0858              6.2791            2.5125                 0.0208        20.2300 0.7071     21.5807     77.7800            8.4851            3.8890            4.9496       31.9402      113.8018         True             True
     2           20 25.0000  60.0000    40.0000      6.0000      5.5000         20.2200          1.0385        0.0824              4.8494            1.7283                 0.0160        20.2300 0.9395     23.4873     56.3695            5.6370            5.1672            6.5764       27.2209       97.6924         True       

In [11]:
# Compare projection-factor distributions and test the specific '20-30% smaller in every dimension' premise.
print('R1 scale summary:',r1geom.scale.describe().to_dict())
print('R2 scale summary:',r2geom.scale.describe().to_dict())
print('R2 counts: 20-30% shrink',((r2geom.scale>=.70)&(r2geom.scale<=.80)).sum(),
      'any shrink',(r2geom.scale<1).sum(),'expanded',(r2geom.scale>1).sum())
# permutation test for round-specific scale distribution difference (means), exact-ish enumeration C(18,9)=48620
from itertools import combinations
vals=np.r_[r1geom.scale.to_numpy(),r2geom.scale.to_numpy()]
obs=r2geom.scale.mean()-r1geom.scale.mean(); dif=[]
inds=np.arange(18)
for ix in combinations(inds,9):
 mask=np.zeros(18,bool); mask[list(ix)]=1
 dif.append(vals[mask].mean()-vals[~mask].mean())
pperm=np.mean(np.abs(dif)>=abs(obs)-1e-12)
print('mean scale R1, R2, difference, exact permutation p:',r1geom.scale.mean(),r2geom.scale.mean(),obs,pperm)
# Measurement ranges of per-drop values for c3 vs all others to show separation/stability
for s in ['r2d2c3','r2d2c2']:
 d=r2drop[r2drop.specimen==s]
 print(s,d[['t180','e_rebound']].agg(['min','max','mean','std']).to_string())
# Is c3 separated drop-by-drop from every other article?
c3=r2drop[r2drop.specimen=='r2d2c3'].t180
others=r2drop[r2drop.specimen!='r2d2c3'].t180
print('c3 min vs others max',c3.min(),others.max())

R1 scale summary: {'count': 9.0, 'mean': 0.8965555555555554, 'std': 0.0930913141908405, 'min': 0.7748, '25%': 0.8497, '50%': 0.8639, '75%': 0.9351, 'max': 1.0437}
R2 scale summary: {'count': 9.0, 'mean': 0.8653333333333334, 'std': 0.12614270688390986, 'min': 0.6732, '25%': 0.8029, '50%': 0.8722, '75%': 0.8985, 'max': 1.092}
R2 counts: 20-30% shrink 1 any shrink 7 expanded 2


mean scale R1, R2, difference, exact permutation p: 0.8965555555555554 0.8653333333333334 -0.03122222222222204 0.5570958453311394
r2d2c3           t180  e_rebound
min   1.303113   0.019406
max   1.346928   0.020412
mean  1.333652   0.020082
std   0.011659   0.000273
r2d2c2           t180  e_rebound
min   1.021611   0.019441
max   1.059980   0.022048
mean  1.041072   0.020954
std   0.015414   0.000800
c3 min vs others max 1.303112765876027 1.2000984974685214


In [12]:
# Quantify pairwise improvement probabilities under the report's stated uncertainty sensitivity.
# Use t180 0.72% floor + within-drop SEM; rebound 5% provisional floor + within-drop SEM.
ix=allx.set_index('specimen')
def p_less(a,b,col,sigcol):
    mu=ix.loc[a,col]-ix.loc[b,col]
    se=np.hypot(ix.loc[a,sigcol],ix.loc[b,sigcol])
    return stats.norm.cdf((0-mu)/se),mu,se
pairs=[('r2d2c1','bpx68c'),('r2d2c6','ajhby6'),('r2d2c6','bpx68c')]
for a,b in pairs:
    pt=p_less(a,b,'t180_mean','t_sig')
    pe=p_less(a,b,'e_reb_mJ','e_abs_sig5')
    print(a,'vs',b,'P lower t',pt,'P lower E',pe,'approx joint dominance',pt[0]*pe[0])
# exact energy percentage reductions
for base in ['ajhby6','bpx68c']:
 print('c6 reduction from',base,(1-ix.loc['r2d2c6','e_reb_mJ']/ix.loc[base,'e_reb_mJ'])*100)
# pooled article-level noise formulas for all objectives and c2 conservative
print('\nNoise floor formulas representative values:')
print(allx[['specimen','t180_mean','t_sig','e_reb_mJ','e_abs_sem','e_abs_sig5']].to_string(float_format=lambda z:f'{z:.5f}'))


r2d2c1 vs bpx68c P lower t (np.float64(0.9510628362036827), np.float64(-0.0169214542786148), np.float64(0.010222915599635974)) P lower E (np.float64(0.6866850632721614), np.float64(-0.20999999999999996), np.float64(0.43167616693971045)) approx joint dominance 0.6530806438543271
r2d2c6 vs ajhby6 P lower t (np.float64(3.332774034346527e-05), np.float64(0.04219094636036358), np.float64(0.010579690544575904)) P lower E (np.float64(0.9999999833797837), np.float64(-2.0299999999999994), np.float64(0.36752382962550734)) approx joint dominance 3.332773978955102e-05
r2d2c6 vs bpx68c P lower t (np.float64(1.7174045866234908e-05), np.float64(0.043777042959432855), np.float64(0.01056768222967392)) P lower E (np.float64(0.9999999938666083), np.float64(-2.1099999999999994), np.float64(0.3704369269951986)) approx joint dominance 1.7174045760899758e-05
c6 reduction from ajhby6 33.27868852459016
c6 reduction from bpx68c 34.14239482200646

Noise floor formulas representative values:
   specimen  t180_mea

In [13]:
# Detector stability summaries and standardized candidate distances for selecting a limited round-3 subset.
r2stab=(r2.assign(e_cv_pct=100*r2.e_rebound_sd/r2.e_rebound_mean,
                  t_cv_pct=100*r2.t180_sd/r2.t180_mean,
                  second_cv_pct=100*r2.t_second_ms_sd/r2.t_second_ms_mean)
        [['specimen','e_rebound_mean','e_rebound_sd','e_cv_pct','t_second_ms_mean','t_second_ms_sd','second_cv_pct','t_cv_pct']])
print('Round-2 rebound/detector stability:')
print(r2stab.to_string(index=False,float_format=lambda x:f'{x:.3f}'))
print('e CV range/median',r2stab.e_cv_pct.min(),r2stab.e_cv_pct.median(),r2stab.e_cv_pct.max())

# Candidate diversity among valid trials in normalized base-design coordinates.
cols5=['R_mm','H_mm','twist_deg','strut_d_mm','cable_d_mm']; lo=np.array([25,60,40,6,3]); hi=np.array([40,110,80,12,5.5])
valid=r3[r3.envelope_ok & r3.cable_bridge_ok].copy()
Z=(valid[cols5].to_numpy()-lo)/(hi-lo)
D=np.sqrt(((Z[:,None,:]-Z[None,:,:])**2).sum(2))
print('\nValid candidate normalized pairwise distances:')
print(pd.DataFrame(D,index=valid.trial_index,columns=valid.trial_index).round(2).to_string())
# Candidate closest to each measured R2 design base coordinate and specifically c6 trial14.
train2=r2geom[cols5].to_numpy(); T=(train2-lo)/(hi-lo)
for i,row in enumerate(valid.itertuples()):
 d=np.sqrt(((T-Z[i])**2).sum(1)); j=d.argmin()
 print('trial',row.trial_index,'nearest measured R2 trial',r2geom.iloc[j].source_trial,'distance',round(d[j],3),
       'pred t',row.pred_t180_mean,'pred E',row.pred_e_reb_mJ_mean)


Round-2 rebound/detector stability:
specimen  e_rebound_mean  e_rebound_sd  e_cv_pct  t_second_ms_mean  t_second_ms_sd  second_cv_pct  t_cv_pct
  r2d2c1           0.021         0.001     2.977            22.303           0.299          1.342     0.207
  r2d2c2           0.021         0.001     3.820            22.725           0.771          3.393     1.481
  r2d2c3           0.020         0.000     1.362            21.738           0.298          1.370     0.874
  r2d2c4           0.022         0.000     0.885            24.179           0.198          0.819     0.640
  r2d2c5           0.019         0.000     1.223            20.925           0.273          1.302     0.346
  r2d2c6           0.014         0.000     1.622            15.701           0.286          1.821     0.401
  r2d2c7           0.044         0.000     0.434            47.716           0.172          0.361     0.088
  r2d2c8           0.026         0.000     1.259            28.497           0.292          1.024   

In [14]:
# Final quantities needed for model-pivot and presentation rulings.
print('Round1 mean/range t:',obj[obj['round']==1].t180_mean.mean(),obj[obj['round']==1].t180_mean.min(),obj[obj['round']==1].t180_mean.max())
print('Round2 mean/range t:',obj[obj['round']==2].t180_mean.mean(),obj[obj['round']==2].t180_mean.min(),obj[obj['round']==2].t180_mean.max())
print('All mean t:',obj.t180_mean.mean(),'R3 predicted mean/range:',r3.pred_t180_mean.mean(),r3.pred_t180_mean.min(),r3.pred_t180_mean.max())
print('Best R1 vs best R2 difference:',obj[obj['round']==2].t180_mean.min()-obj[obj['round']==1].t180_mean.min())
print('R2 articles below 1:',(obj[obj['round']==2].t180_mean<1).sum())
# Calibration with intervals explicitly tabulated.
cal=merged[['print_id','t180_mean','pred_t180_mean','pred_t180_sd','t180_resid','t180_z','meas_e_mJ_calc','pred_e_reb_mJ_mean','pred_e_reb_mJ_sd','e_reb_mJ_resid','e_reb_mJ_z']].copy()
cal['t_in68']=cal.t180_z.abs()<=1; cal['t_in95']=cal.t180_z.abs()<=1.96
cal['e_in68']=cal.e_reb_mJ_z.abs()<=1; cal['e_in95']=cal.e_reb_mJ_z.abs()<=1.96
print(cal.to_markdown(index=False,floatfmt='.3f'))
# Candidate selected set idea: 19 exploitation-ish t, 20 low E, 24 interior/less redundant. All valid.
print('\nSelected proposed candidates')
print(r3[r3.trial_index.isin([19,20,24])][['trial_index']+cols5+['pred_t180_mean','pred_t180_sd','pred_e_reb_mJ_mean','pred_e_reb_mJ_sd','envelope_ok','cable_bridge_ok']].to_markdown(index=False,floatfmt='.3f'))

Round1 mean/range t: 1.0077188521806408 0.8930777877843858 1.0616197738255833
Round2 mean/range t: 1.092416648947592 0.9483789100007614 1.3336523122690265
All mean t: 1.0525588622337327 R3 predicted mean/range: 1.0207111111111111 0.9976 1.0521
Best R1 vs best R2 difference: 0.05530112221637551
R2 articles below 1: 2
| print_id   |   t180_mean |   pred_t180_mean |   pred_t180_sd |   t180_resid |   t180_z |   meas_e_mJ_calc |   pred_e_reb_mJ_mean |   pred_e_reb_mJ_sd |   e_reb_mJ_resid |   e_reb_mJ_z | t_in68   | t_in95   | e_in68   | e_in95   |
|:-----------|------------:|-----------------:|---------------:|-------------:|---------:|-----------------:|---------------------:|-------------------:|-----------------:|-------------:|:---------|:---------|:---------|:---------|
| r2d2c1     |       0.994 |            0.948 |          0.046 |        0.046 |    1.001 |            5.965 |                8.307 |              3.176 |           -2.342 |       -0.737 | False    | True     | True    

In [15]:
# Inspect round-2 base designs and isolate matched/near-matched comparisons relevant to attribution.
base=r2geom[['source_trial','R_mm','H_mm','twist_deg','strut_d_mm','cable_d_mm','scale','R_print_mm','H_print_mm','strut_d_print_mm','cable_d_print_mm']].merge(key[['source_trial','print_id']],on='source_trial').merge(r2[['specimen','t180_mean','t1000_mean','e_rebound_mean','mass_g']],left_on='print_id',right_on='specimen')
print(base.sort_values('source_trial').to_string(index=False,float_format=lambda x:f'{x:.4f}'))
# c3 vs c6 differ only twist among base coords?
a=base.set_index('print_id')
print('\nc3-c6 base coordinate differences:')
print((a.loc['r2d2c3',['R_mm','H_mm','twist_deg','strut_d_mm','cable_d_mm']]-a.loc['r2d2c6',['R_mm','H_mm','twist_deg','strut_d_mm','cable_d_mm']]).to_string())
print('c3,c6 outcomes:\n',a.loc[['r2d2c3','r2d2c6'],['t180_mean','t1000_mean','e_rebound_mean','mass_g','scale','R_print_mm','H_print_mm','strut_d_print_mm','cable_d_print_mm']].to_string())

 source_trial    R_mm     H_mm  twist_deg  strut_d_mm  cable_d_mm  scale  R_print_mm  H_print_mm  strut_d_print_mm  cable_d_print_mm print_id specimen  t180_mean  t1000_mean  e_rebound_mean  mass_g
           10 25.0000  60.0000    80.0000     12.0000      3.0000 0.8779     21.9460     52.6710           10.5340            2.6340   r2d2c9   r2d2c9     1.0530      1.4741          0.0218 17.9100
           11 40.0000  60.0000    80.0000     12.0000      5.5000 0.7531     30.1240     45.1860            9.0370            4.1420   r2d2c4   r2d2c4     1.1847      1.2213          0.0223 19.3300
           12 25.0000  60.0000    40.0000     12.0000      3.0000 0.8985     22.4630     53.9120           10.7820            2.6960   r2d2c2   r2d2c2     1.0411      1.1142          0.0210 17.9600
           13 25.0000  60.0000    80.0000      7.4015      3.0000 1.0920     27.3010     65.5220            8.0830            3.2760   r2d2c1   r2d2c1     0.9942      1.0024          0.0206 19.3800
          

In [16]:
# Last calibration discrimination: did frozen predictions rank the nine outcomes?
for obs,pm in [('t180_mean','pred_t180_mean'),('meas_e_mJ_calc','pred_e_reb_mJ_mean')]:
    print(obs,'Pearson',stats.pearsonr(merged[pm],merged[obs]),'Spearman',stats.spearmanr(merged[pm],merged[obs]))
# Bias CIs across designs, descriptive t interval and nonparam bootstrap percentile.
rng=np.random.default_rng(20260825)
for m in ['t180_resid','e_reb_mJ_resid']:
 x=merged[m].to_numpy(); ci=stats.t.interval(.95,len(x)-1,loc=x.mean(),scale=stats.sem(x))
 boot=np.mean(rng.choice(x,(200000,len(x)),replace=True),axis=1)
 print(m,'mean',x.mean(),'tCI',ci,'bootstrap percentile',np.quantile(boot,[.025,.975]))


t180_mean Pearson PearsonRResult(statistic=np.float64(-0.0779000188976165), pvalue=np.float64(0.8421037251872528)) Spearman SignificanceResult(statistic=np.float64(0.11666666666666665), pvalue=np.float64(0.765007942926146))
meas_e_mJ_calc Pearson PearsonRResult(statistic=np.float64(0.6171903289757706), pvalue=np.float64(0.07660669704961864)) Spearman SignificanceResult(statistic=np.float64(0.3), pvalue=np.float64(0.43284532670948234))
t180_resid mean 0.1656499822809253 tCI (np.float64(0.07036149652629348), np.float64(0.26093846803555715)) bootstrap percentile [0.09654848 0.24850418]
e_reb_mJ_resid mean -3.54381155195224 tCI (np.float64(-4.941491361522242), np.float64(-2.1461317423822384)) bootstrap percentile [-4.64668438 -2.41248646]


In [17]:
from pathlib import Path
outdir=Path('/workspace/edison-trajectories/round2-results')
outdir.mkdir(parents=True,exist_ok=True)
report=r'''# Follow-up adversarial review of round-2 results

**Review date:** 2026-08-25  
**Decision for the 8/25 presentation:** Show that round 2 produced a reproducible empirical extension of the measured two-metric set, especially `r2d2c6` at 4.07 mJ, but do not call this a validated engineering Pareto improvement. The frozen model failed calibration in opposite directions on both outputs. The data establish model misspecification; they do not identify reprojection stiffness as its cause.  
**Decision for round 3:** Do not print the nine proposed articles as a Bayesian-optimization batch. Refit on as-printed geometry, use `t180` as the sole objective with article-level noise, and spend nine print slots on three independently printed articles at each of three geometries: `6lhxfy`, `r2d2c6`, and one newly regenerated design. Re-seat and re-test `r2d2c3` before it enters the fit.

## What changes what you present and print

1. **Correct “8 of 9 rebound values were at or below prediction” to “9 of 9.”** All nine `t180` residuals were positive and all nine rebound-energy residuals were negative. Both sign tests give exact two-sided $p=0.0039$. This is not calibrated uncertainty centered on reality.
2. **Do not present reprojection stiffness as the explanation.** It is a plausible hypothesis, not an identified mechanism. The premise that all articles were 20–30% smaller is not reproduced: seven of nine were shrunk, two were expanded, and only one had scale 0.70–0.80. Round-2 mean scale was 0.865 versus 0.897 in round 1, an unremarkable 3.1 percentage-point difference (exact permutation $p=0.557$). More shrink did not significantly predict larger `t180` error ($r=0.382$, $p=0.310$).
3. **Replace “the uncertainty bands did their job” with an objective-specific calibration statement.** Rebound had 9/9 coverage at nominal 95%, but all observations were on the same side and the bands were very wide. `t180` had only 5/9 coverage at 95% and 1/9 at 68%. Posterior standard deviations of a noise-free mean are not full predictive intervals for a newly printed article.
4. **Keep `e_rebound` as a diagnostic, not a BO objective.** Round 2 materially improves detector-stability evidence, but not event identity or payload relevance. The delayed-event metric may become a constraint after synchronized video and a response-amplitude criterion validate what is being constrained.
5. **Do not use the present round-3 suggestions as evidence for a low-twist optimum.** Eight of nine are at 40° twist and seven of nine at 5.5 mm cables; two also fail stated print constraints. This is another boundary-heavy batch from the same bad likelihood and an altered representation.

# A. Calibration audit

## A1. Recomputed prediction errors

I joined `round2-frozen-predictions.csv` to `round2-print-key.csv` by trial and then to `round2-measured-drop-results.csv` by print ID. I independently reconstructed energy as

$$E_{reb}=e_{rebound}mgh,$$

using $g=9.80665\ \mathrm{m/s^2}$ and $h=1.524\ \mathrm{m}$. The largest discrepancy from `objectives-mass-normalized.csv` was 0.0049 mJ, attributable to rounding. Standardized residuals below use the requested frozen posterior standard deviation of the latent, noise-free mean:

$$z_i=\frac{y_i-\hat\mu_i}{\hat\sigma_i}.$$

| Article | `t180` measured | predicted ± SD | residual | z | rebound measured (mJ) | predicted ± SD (mJ) | residual (mJ) | z |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| `r2d2c1` | 0.994 | 0.948 ± 0.046 | +0.046 | +1.00 | 5.97 | 8.31 ± 3.18 | −2.34 | −0.74 |
| `r2d2c2` | 1.041 | 0.926 ± 0.095 | +0.115 | +1.22 | 5.62 | 9.90 ± 5.03 | −4.27 | −0.85 |
| `r2d2c3` | 1.334 | 0.910 ± 0.073 | +0.423 | +5.82 | 5.59 | 10.09 ± 5.20 | −4.50 | −0.87 |
| `r2d2c4` | 1.185 | 0.885 ± 0.075 | +0.300 | +4.00 | 6.44 | 12.25 ± 5.73 | −5.80 | −1.01 |
| `r2d2c5` | 1.152 | 0.987 ± 0.066 | +0.165 | +2.48 | 6.71 | 8.37 ± 4.34 | −1.67 | −0.38 |
| `r2d2c6` | 1.055 | 0.953 ± 0.094 | +0.102 | +1.08 | 4.07 | 8.98 ± 5.58 | −4.91 | −0.88 |
| `r2d2c7` | 0.948 | 0.905 ± 0.091 | +0.043 | +0.47 | 12.17 | 13.02 ± 5.11 | −0.86 | −0.17 |
| `r2d2c8` | 1.070 | 0.955 ± 0.071 | +0.116 | +1.63 | 7.89 | 9.97 ± 5.34 | −2.08 | −0.39 |
| `r2d2c9` | 1.053 | 0.873 ± 0.053 | +0.180 | +3.43 | 5.84 | 11.31 ± 4.29 | −5.47 | −1.27 |

### Batch diagnostics

| Diagnostic | `t180` | rebound energy |
|---|---:|---:|
| Mean residual | +0.166 | −3.54 mJ |
| Mean standardized residual | +2.35 | −0.729 |
| SD of standardized residuals | 1.76 | 0.351 |
| Nominal 68% coverage | 1/9 (11%) | 7/9 (78%) |
| Nominal 95% coverage | 5/9 (56%) | 9/9 (100%) |
| Residual signs | 9 positive, 0 negative | 0 positive, 9 negative |
| Exact two-sided sign-test $p$ | 0.0039 | 0.0039 |
| Frozen prediction versus outcome, Pearson $r$ | −0.078 ($p=0.842$) | +0.617 ($p=0.077$) |
| Frozen prediction versus outcome, Spearman $\rho$ | +0.117 ($p=0.765$) | +0.300 ($p=0.433$) |

For context, exact binomial 95% confidence intervals around the empirical 95% coverage rates are 0.212–0.863 for `t180` and 0.664–1.000 for rebound. Nine points cannot estimate a smooth calibration curve. They can expose gross directional bias, which is what happened here.

## A2. Attribution: established misspecification, unresolved mechanism

**Ruling: both noise misspecification and representation shift remain plausible, but the data identify neither as the unique cause. The only firm attribution is model misspecification under deployment covariates.**

What the results establish:

- The frozen model was systematically optimistic for `t180` and systematically high for rebound. Mean residual 95% t intervals across the nine designs were +0.070 to +0.261 for `t180` and −4.94 to −2.15 mJ for rebound.
- It had no ranking skill for `t180` in this batch. That is stronger evidence than simple undercoverage.
- Per-drop SEM was the wrong variance for predicting a new article. It can cause short length scales, excessive confidence, and aggressive acquisition, but **an understated zero-mean noise variance does not by itself predict that every residual will have the same sign**. It explains overconfidence better than directional bias.
- A deployment shift can create directional bias, but the supplied geometry does not isolate it. Round 1 was also projected before printing, no base design was printed at two controlled scales, and scale is confounded with all five design coordinates.

Evidence against presenting shrink as the identified cause:

- Round-2 scales ranged from 0.673 to 1.092. Seven articles shrank and two expanded. The claimed universal 20–30% shrink is false for these files.
- Scale distributions overlap strongly between rounds. Mean scale changed from 0.897 in round 1 to 0.865 in round 2.
- Within round 2, shrink percentage versus `t180` residual gave Pearson $r=0.382$ ($p=0.310$) and Spearman $\rho=0.400$ ($p=0.286$). The corresponding correlations with rebound residual were −0.326 ($p=0.391$) and −0.300 ($p=0.433$).
- The two residuals were negatively associated, as the stiffness story predicts, but weakly: Pearson $r=-0.555$ ($p=0.121$). This is suggestive, not discriminating.

Other plausible contributors are the unmodeled projection mapping itself, extrapolation from only eight mapped round-1 articles, print/mount realization, detector definition, defects in three round-2 articles, and using base coordinates when the tested objects occupy different absolute dimensions. The data do not apportion these causes.

**Round-3 implication:** fix both failures. Use article-level noise and train/generate in the same as-printed representation. Do not choose one remedy based on this batch.

## A3. Scorecard against the 8/21 review

| Prior prediction or conclusion | Result |
|---|---|
| Per-drop SEM would make the optimizer overconfident for new articles. | **Supported for `t180`.** Only 5/9 fell inside nominal 95% latent-mean bands, and three exceeded 3 SD. Rebound bands were wide rather than overconfident. |
| Boundary collapse was not evidence of an optimum. | **Supported.** No round-2 article beat `6lhxfy` on `t180`; the frozen `t180` means had no within-batch ranking skill. |
| qNEHVI was extending a fragile inferred front. | **Partly supported.** The observed empirical front did extend, especially on the low-rebound channel, but not with a validated second objective or article replication. The prior review was too dismissive about the possibility of useful rebound discrimination. |
| `e_rebound` detector was fragile across new geometry. | **Refuted for this batch.** All nine yielded stable values, with within-session rebound CV 0.43–3.82% (median 1.26%) and second-event-time CV 0.36–3.39%. This is meaningful detector evidence. It does not validate event identity. |
| `e_rebound` should not be an objective without physical validation. | **Still supported.** Stable detection answers repeatability within these sessions, not what moved, whether the event is harmful, or whether minimizing flight-time-derived velocity is the correct direction. |
| A batch of one article per design cannot estimate design repeatability. | **Supported and now directly limiting.** No print-to-print variance is available for any new front point. |
| Article-level noise alone explained the collapse. | **The prior review did not claim uniqueness and was right not to.** The new data elevate representation shift as a competing explanation that the prior review did not analyze. |

# B. The rebound channel after round 2

## B1. Status of the metric

Round 2 supplies two real advances:

1. The detector returned stable delayed-event values across nine new geometries, including a 3-fold range in article means from 0.0142 to 0.0440.
2. `r2d2c6` produced a channel value of 4.07 mJ versus the previous absolute-energy floor of 6.10–6.18 mJ, a 33–34% decrease.

That moves `e_rebound` from “fragile diagnostic” to **stable diagnostic in this batch**. It does not make it a valid engineering objective. The formula still estimates a velocity ratio under ballistic assumptions and is then multiplied by $mgh$ without squaring the velocity ratio. It is not measured rebound energy. No synchronized video, relative-motion record, or second-impact amplitude establishes event identity or damage relevance.

Promotion path:

- **To a constraint:** synchronized high-speed video must show that the same detected interval consistently brackets separation and re-contact across at least a high-hop and low-hop geometry, restrained/unrestrained intervention must alter the event as predicted, and the constraint must be stated on a payload-relevant second-impact peak or shock-response-spectrum ordinate. Independently printed articles must reproduce it.
- **To an objective:** all constraint requirements plus an engineering utility showing why lower is monotonically better. The present data do not supply that utility.

## B2. Correlation and whether a front is established

There are 17 mapped articles in `objectives-mass-normalized.csv`, not 18. The eighteenth tested article is unmapped `amdjwm` and correctly cannot enter a design-space or Pareto analysis.

Across the 17 mapped articles:

- `t180` versus absolute rebound energy: Pearson $r=-0.537$, $p=0.026$; Spearman $\rho=-0.456$, $p=0.066$; Kendall $\tau=-0.338$, $p=0.063$.
- Excluding `6lhxfy`: Pearson $r=-0.401$, $p=0.124$; Spearman $\rho=-0.347$, $p=0.188$.
- Round 2 alone: Pearson $r=-0.399$, $p=0.287$; Spearman $\rho=-0.217$, $p=0.576$.
- A 200,000-resample article bootstrap gave a descriptive Pearson 95% interval of −0.785 to −0.166. This interval does not correct measurement error or establish a mechanism.

The anti-association is less dependent on one point than at $n=7$, but rank evidence remains marginal and round 2 by itself does not show a monotonic trade-off. An **empirical nondominated set exists by definition**. A stable physical Pareto front does not yet exist as an evidence-backed claim. A one-dimensional compliance mechanism remains compatible with the data.

## B3. Absolute versus per-gram framing

The recomputed fronts exactly match the flags:

- Absolute mJ: `{6lhxfy, r2d2c7, r2d2c1, r2d2c2, r2d2c6}`.
- Per gram: `{6lhxfy, r2d2c7, r2d2c1, ajhby6, bpx68c, r2d2c6}`.
- Common set: `{6lhxfy, r2d2c7, r2d2c1, r2d2c6}`.

The team’s common-membership claim survives. `r2d2c2` is not robust to framing and is drift-contaminated. Present the **per-gram form as a rescaled diagnostic velocity ratio**, not as specific energy absorption. It removes specimen mass from a quantity whose physical “energy” interpretation is already unsupported. If stakeholders require the historical BO plot, show absolute mJ beside it and state that the front changes.

# C. Rulings on proposed presentation claims

## C1. “The batch genuinely improved the Pareto front…”

**Verdict: survives with a required caveat.** The arithmetic is correct for measured means: four of five absolute-front points are round 2; `r2d2c1` is lower than `bpx68c` on both means; and `r2d2c6` lowers the measured absolute floor from 6.10–6.18 to 4.07 mJ.

Use this caveat verbatim:

> **“This is an empirical front of single printed articles on a physically unvalidated rebound proxy. It is not yet a replicated design-level Pareto front; `r2d2c2` is drift-contaminated, and front membership changes under mass normalization.”**

Do not say `r2d2c1` “strictly dominates” `bpx68c` without “on observed means.” With the 0.72% `t180` floor and a provisional 5% rebound floor, the approximate probability that `r2d2c1` is lower on both is only 0.65. By contrast, `r2d2c6`’s low-rebound separation is large: 4.07 versus 6.10–6.18 mJ.

## C2. “The model was systematically optimistic… plausible physics…”

**Verdict: survives with a required caveat for the first clause; does not survive as a causal explanation.**

Use this caveat verbatim:

> **“All nine `t180` outcomes exceeded their frozen predictions, but these data do not identify why. Reprojection-induced stiffness is one hypothesis; noise misspecification, sparse extrapolation, mounting, defects, and the mismatch between base and as-printed coordinates remain confounded.”**

The sentence “the same stiffness that raises `t180` lowers restitution” is consistent with the signs but is a just-so story at present. Scale did not significantly explain residual magnitude, and there is no controlled same-design/different-scale comparison.

## C3. “The uncertainty bands did their job…”

**Verdict: does not survive.** It is selective. A fair slide statement is:

> **“The wide rebound bands covered all nine measurements but were directionally biased: all nine outcomes were below their predicted means. `t180` was miscalibrated, with only 5/9 inside nominal 95% latent-mean bands.”**

## C4. Additional result worth presenting

Say this:

> **“Round 2 proved that the rebound detector can be stable and design-discriminating within short sessions across nine new geometries, but one article per geometry cannot separate design effects from print and mounting realization.”**

Also say that the frozen model did not improve the primary endpoint: best round-2 `t180` was 0.948 (`r2d2c7`), 0.055 above round-1 best `6lhxfy` at 0.893. Only two of nine round-2 articles had `t180<1`.

# D. Data-quality rulings

## D1. `r2d2c3`

The recorded result is internally stable but physically ambiguous:

- Every scored drop was separated from every other article: `r2d2c3` minimum `t180` was 1.303; the maximum among the other eight was 1.200.
- Its linear drift was −0.0145% per drop ($p=0.703$), so the high mean is not a session ramp.
- `t1000/t180=1.860`, versus 0.999–1.450 elsewhere, is a real signal-processing signature, not random scatter.
- `r2d2c3` and `r2d2c6` have the same base $R,H,$ strut, and cable settings and nearly identical as-printed dimensions; twist differs by 40°. Their large outcome difference could therefore be a real twist-controlled broadband mode. It could also be seating/mount coupling because both `t180` and especially `t1000` are mount-sensitive.

The committed metrics cannot discriminate those explanations. **Hold `r2d2c3` out of training pending re-test.** Do not merely deweight a possible structural outlier into a possible mount outlier.

Minimal re-test: remove and reinstall the accelerometer/mount, document seating, then collect two blinded randomized blocks of 10–12 stabilized drops on the same article, interleaved with `r2d2c6` or `bpx68c`. If the 1.33/2.48 signature survives independent reseating, ingest both sessions with a session effect. If it collapses, mark the first session as a mount failure.

## D2. `r2d2c2`

The drift is not subtle: slope +0.2529% per drop, $p=6.0\times10^{-11}$, with first-five versus last-five means 1.023 and 1.059. **Re-test and exclude the current mean from front and GP claims until a clean session exists.** An illustrative uncertainty that adds half the 3.5% excursion in quadrature is 0.020, versus 0.0083 from the 0.72% floor plus drop SEM, but that inflation assumes an arbitrary drift distribution and does not repair a window-dependent estimand.

## D3. Mixed session lengths

Per-drop SEM is not adequate. It addresses uncertainty in an article’s session mean, not print-to-print prediction. In round 1, first-19 versus full-session differences reached 0.00793 in `t180` (0.747%), essentially the entire 0.72% article floor. For rebound they reached 21.1% for `amdjwm`; even excluding that detector failure, `ajhby6` shifted 3.91%.

For the immediate refit, re-window round 1 to the first 19 stabilized drops so both rounds estimate the same early-session mean. Retain full data for diagnostics. The better later model is a hierarchical drop-index/session model rather than discarding data. Use randomized blocks and a reference article so time drift is estimable rather than silently folded into geometry.

# E. Round-3 ruling

## E1. Diagnosis of the proposed pivot

The new batch is not evidence of genuine low-twist learning. Eight of nine points are at minimum twist and seven at maximum cable diameter. Trial 22 violates the envelope limit and trial 26 violates the cable-bridge rule. The proposed `t180` means, 0.998–1.052 with mean 1.021, sit near the mapped round-1 mean 1.008 and all-data mean 1.053. This is compatible with posterior regression toward the data center after the optimistic corner failed. It is not evidence that the model learned an improved attenuator: no proposed mean approaches observed `6lhxfy=0.893`, and no round-2 article beat it.

The low-rebound side is also tracking a single unreplicated observation: trials 20 and 25 are predicted near 4.85 mJ after `r2d2c6=4.07` mJ. Calling this a learned corner would be premature.

## E2. Concrete formulation and allocation

### Model

- **Objective:** minimize `t180` only, interpreted as a CFC-180 peak-ratio screening endpoint.
- **Exposure controls:** restrict test sessions to the demonstrated healthy input-$\Delta v$ band and include centered session-mean `in_dv_ms` as a nuisance covariate. Record `in_180_g`, pulse width, baseline quality, saturation, block, and reference response. Do not optimize exposure.
- **Diagnostics:** raw `out_180_g`, `t1000`, delayed-event timing and amplitude, measured mass, print defects, and modal-fit outputs where valid.
- **Constraints:** printed mass target 20.23 g with a prespecified manufacturing tolerance based on achieved process capability, envelope ≤250 cm³, cable bridge ≥3.0 mm, no invalid captures, and no unresolved drift flag. Do not use the artificial ±0.01 g optimizer slab as if it were physical process capability.
- **`t180` observation SD:**

$$\sigma_{T,i}=\sqrt{(0.0072y_i)^2+\frac{s_{drop,i}^2}{n_i}}.$$

  Across the supplied mapped articles this is about 0.0064–0.0100; for ordinary, unflagged round-2 articles it is about 0.0068–0.0087. Run sensitivity fits at 0.5%, 1%, and 2% relative floors. Handle `r2d2c2` and `r2d2c3` by data-quality rulings, not arbitrary variance inflation.
- **Rebound, if shown in a non-acquisition sensitivity fit only:**

$$\sigma_{E,i}=\sqrt{(0.05E_i)^2+SEM_{drop,E,i}^2}.$$

  This gives approximately 0.20–0.70 mJ over the mapped data. The 5% term is a conservative sensitivity choice from prior session evidence, not an estimated independent-print SD. Do not use rebound in acquisition until validation.
- **Acquisition:** compare corrected single-objective SAASBO with a regularized conventional GP and a space-filling or Thompson-sampling batch over several seeds. A new geometry should print only if its selection is robust to the 0.5–2% noise range and model choice. qNEHVI is not applicable with one objective; noisy expected improvement or Thompson sampling is.

This follows the distinction between replication noise and exploration used in stochastic Gaussian-process design [Binois et al., 2018](https://doi.org/10.1080/10618600.2018.1458625); [Binois et al., 2019](https://doi.org/10.1080/00401706.2018.1469433). qNEHVI accounts only for uncertainty supplied to its model and does not protect against a wrong objective, likelihood, or covariate representation [Daulton et al., 2021](https://doi.org/10.48550/arXiv.2105.08195). SAASBO’s sparse-axis prior does not remove the need for adequate replication or deployment-valid inputs [Eriksson and Jankowiak, 2021](https://doi.org/10.48550/arXiv.2103.00349).

### Search representation

Refit on **as-printed** `R`, `H`, strut diameter, cable diameter, twist, and measured mass. Generate candidates through the same print-projection function first, then evaluate acquisition at their projected coordinates. Treat measured mass as a post-print covariate/constraint, not a freely exploitable design coordinate.

Printed mass alone does not absorb the representation shift. Two objects can have the same mass and different absolute dimensions, strut slenderness, cable section, and material distribution. A model fitted to base coordinates plus mass still asks the GP to infer an unobserved nonlinear projection from 17 articles. A projection-regime indicator cannot be estimated before any outcomes exist under the new regime. The clean bridge is to reproduce known as-printed geometries under controlled manufacture and ingest realized geometry and mass.

### Nine print slots

Print **three independently manufactured articles at each of three geometries**:

1. Three at the as-printed `6lhxfy` geometry, the unreplicated primary-endpoint best.
2. Three at the as-printed `r2d2c6` geometry, the unreplicated low-rebound extreme.
3. Three at one newly regenerated, constraint-valid geometry selected by agreement among corrected fits.

Use 10–12 stabilized drops per article after two warm-ups, randomized in blocks, with the same durable reference article at block start and end. The existing originals should not be counted as exchangeable replicates if process/projection conditions differ.

**Print decision:** regenerate. Print none of trials 19–27 as a BO recommendation. If schedule makes partial use unavoidable before refitting, trials 19, 20, and 24 are the only defensible sentinel subset because they are constraint-valid and span distinct corners; label them “model-diagnostic sentinels,” not optimized round-3 designs. Do not print trials 22 or 26.

## E3. Does the mass parameter fix the shift?

No. It helps condition on one important realized property, but it cannot encode the change from constant-solid-mass scaling to constant-printed-mass manufacture. The corrective action is as-printed geometry for fitting and generation, measured mass as a covariate/constraint, and matched replication bridging the manufacturing regimes. Any prediction under the new projection policy remains an extrapolation until such bridge data exist.

# F. Final verdict

1. **Calibration attribution:** the frozen model is decisively directionally biased on both outputs. Noise misspecification explains false precision; a base-versus-as-printed representation mismatch can explain deployment bias. The present data do not separate them, and the claimed universal 20–30% shrink is not in the numeric files. Fix both.
2. **Presentation claims:** C1 survives only as an observed, unreplicated proxy-front claim with the verbatim caveat above. C2’s optimism survives, but the stiffness attribution does not. C3 does not survive. Add the detector-stability result and the failure to improve best `t180`.
3. **Round 3:** one objective (`t180`), article-level 0.72% CV floor plus drop SEM, as-printed geometry, mass as realized covariate/constraint, input severity as nuisance covariate, and three articles each at `6lhxfy`, `r2d2c6`, and one regenerated design. Do not print the nine proposed candidates as a batch.
4. **Cheapest measurement with the highest immediate decision value:** **re-seat and re-test `r2d2c3`.** It determines whether the largest calibration failure and strongest broadband anomaly belongs in the 17-point fit. One article, two short blocks, and no new print are required. A restrained/unrestrained video test remains the cheapest way to validate rebound physics, but it does not resolve the immediate training-set contamination that will determine round-3 geometry.

# Reproducibility notes and limitations

- All summary means and SDs reproduced exactly from the 172 stabilized round-2 rows. The CSV field `n_valid` counts 21–22 trigger-valid captures, whereas the reported means use 19–20 rows after two warm-ups. Noise calculations in this review use the actual scored counts.
- `objectives-mass-normalized.csv` contains 17 mapped articles. Text calling this an 18-article objective analysis includes unmapped `amdjwm`; it cannot be used in a geometry-conditioned fit.
- There are no raw acceleration time histories in this review bundle. I could not recompute filtering, event picking, second-impact amplitude, shock-response spectra, or mount coupling. SAE J211 supports impact-channel filtering but does not validate the physical interpretation of this ratio [SAE J211/1_202208](https://doi.org/10.4271/J211/1_202208). Shock-response-spectrum validation would require the time histories and a specified damping/frequency range (ISO 18431-4:2007).
- Pareto probabilities are sensitivity calculations, not inferential proof. They assume independent normal article errors, a 0.72% `t180` floor, and a provisional 5% rebound floor. No independent-print rebound variance has been measured.
- Correlations are observational across selected designs. They do not establish causation or a population Pareto surface.

# Discretionary analytical decisions

- Treated the independently printed article, not a drop, as the design-level replication unit.
- Used two-sided exact sign tests at $\alpha=0.05$ and reported nominal 68% and 95% coverage against frozen latent-mean posterior SDs.
- Used both Pearson and rank correlations, leave-one-out sensitivity, and a 200,000-resample article bootstrap because one-point leverage remained plausible.
- Used the 0.72% CV as the central `t180` article-noise floor, with 0.5%, 1%, and 2% sensitivity fits.
- Used a 5% relative rebound floor only for sensitivity calculations; it is not claimed as an estimated print variance.
- Recomputed short-session comparability at the first 19 stabilized drops, matching the modal round-2 scored count.
- Recommended holding `r2d2c3` and the current `r2d2c2` session out rather than assigning subjective large variances.
- Chose a three-geometry by three-print allocation to estimate article repeatability at both observed extremes while retaining one corrected-model exploration geometry.
- Preferred the per-gram rebound display because it removes measured specimen mass, while explicitly refusing to call it specific absorbed energy.
'''
path=outdir/'prior-edison-adversarial-objective-review-followup.md'
path.write_text(report)
print(path, path.stat().st_size, 'bytes', len(report.splitlines()), 'lines')
# lightweight content checks
banned=[' moreover ',' delve ',' embark ','—','~~']
for b in banned:
    print(repr(b), report.lower().count(b))


/workspace/edison-trajectories/round2-results/prior-edison-adversarial-objective-review-followup.md 27917 bytes 266 lines
' moreover ' 0
' delve ' 0
' embark ' 0
'—' 0
'~~' 0


In [18]:
from pathlib import Path
p=Path('/workspace/edison-trajectories/round2-results/prior-edison-adversarial-objective-review-followup.md')
s=p.read_text()
s=s.replace('Show that round 2 produced a reproducible empirical extension of the measured two-metric set, especially `r2d2c6` at 4.07 mJ, but do not call this a validated engineering Pareto improvement.', 'Show that round 2 produced an empirical extension of the measured two-metric set, especially `r2d2c6` at 4.07 mJ, but do not call this a replicated or validated engineering Pareto improvement.')
p.write_text(s)
print(p)
print('bytes:',p.stat().st_size,'lines:',len(s.splitlines()))
print('updated phrase present:', 'replicated or validated engineering Pareto improvement' in s)

/workspace/edison-trajectories/round2-results/prior-edison-adversarial-objective-review-followup.md
bytes: 27919 lines: 266
updated phrase present: True


In [19]:
import os
from e14c.data_storage import upload
path='/workspace/edison-trajectories/round2-results/prior-edison-adversarial-objective-review-followup.md'
uri=upload(path, description='Self-contained adversarial follow-up review of measured round-2 outcomes, calibration, presentation claims, data quality, and concrete round-3 print prescription.', name='prior-edison-adversarial-objective-review-followup.md', project_id=os.environ.get('PROJECT_ID'))
print(uri)

Uploaded /workspace/edison-trajectories/round2-results/prior-edison-adversarial-objective-review-followup.md to data storage as entry data_entry:prior-edison-adversarial-objective-review-follo-4odm under the name 'prior-edison-adversarial-objective-review-followup.md'.
data_entry:prior-edison-adversarial-objective-review-follo-4odm
